In [ ]:
# ============================================
# 最重要：必须在 import torch 之前设置
# ============================================
import os

# 使用单个GPU（显存最大的那个）
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # 使用GPU 0
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 限制PyTorch内存使用
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

# 导入 PyTorch 和其他库
import gc
import pickle
import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, SAGEConv, GATConv, GATv2Conv
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.neighbors import kneighbors_graph
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    precision_recall_fscore_support,
    roc_auc_score,
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from itertools import product
from datetime import datetime
from openpyxl import load_workbook

warnings.filterwarnings("ignore")

# ============================================
# 0. 统一路径管理配置
# ============================================
print("\n" + "=" * 60)
print("0. 初始化实验目录结构")
print("=" * 60)

# 定义基础路径
BASE_DIR = "./"
MODELS_DIR = os.path.join(BASE_DIR, "models")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
FIGURES_DIR = os.path.join(BASE_DIR, "figures")
CURVES_DIR = os.path.join(FIGURES_DIR, "training_curves")
CONFUSION_DIR = os.path.join(FIGURES_DIR, "confusion_matrices")

# 创建主目录
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(CURVES_DIR, exist_ok=True)
os.makedirs(CONFUSION_DIR, exist_ok=True)

# 定义各模型专属目录
MODEL_DIRS = {
    "MLP": os.path.join(MODELS_DIR, "MLP"),
    "GCN": os.path.join(MODELS_DIR, "GCN"),
    "GraphSAGE": os.path.join(MODELS_DIR, "GraphSAGE"),
    "GAT": os.path.join(MODELS_DIR, "GAT"),
    "GATv2": os.path.join(MODELS_DIR, "GATv2"),
}

# 创建各模型目录
for model_dir in MODEL_DIRS.values():
    os.makedirs(model_dir, exist_ok=True)

# 定义Excel文件路径（每个模型一个Excel文件）
EXCEL_PATHS = {
    "MLP": os.path.join(MODEL_DIRS["MLP"], "MLP_results.xlsx"),
    "GCN": os.path.join(
        MODEL_DIRS["GCN"], "GCN_results.xlsx"
    ),
    "GraphSAGE": os.path.join(MODEL_DIRS["GraphSAGE"], "GraphSAGE_results.xlsx"),
    "GAT": os.path.join(MODEL_DIRS["GAT"], "GAT_results.xlsx"),
    "GATv2": os.path.join(MODEL_DIRS["GATv2"], "GATv2_results.xlsx"),
}

# 定义图片保存路径
FIGURE_PATHS = {
    "validation_comparison": os.path.join(
        FIGURES_DIR, "model_validation_comparison.png"
    ),
    "test_comparison": os.path.join(FIGURES_DIR, "model_test_accuracy_comparison.png"),
    "f1_comparison": os.path.join(FIGURES_DIR, "model_f1_comparison.png"),
    "parameter_analysis": os.path.join(FIGURES_DIR, "parameter_analysis.png"),
}

print(f"✓ 主目录: {BASE_DIR}")
print(f"✓ 模型目录: {MODELS_DIR}")
print(f"✓ 结果目录: {RESULTS_DIR}")
print(f"✓ 图片目录: {FIGURES_DIR}")
for model_name, model_dir in MODEL_DIRS.items():
    print(f"✓ {model_name}目录: {model_dir}")

In [ ]:
# ============================================
# 辅助函数：保存模型和结果
# ============================================


def save_model_and_results(model, config, model_name, training_history=None):
    """
    保存模型到文件，并追加结果到Excel

    参数:
        model: 训练好的模型
        config: 配置字典，包含所有参数和指标
        model_name: 模型名称
        training_history: 训练历史（损失、准确率等）
    """

    # 生成模型文件名
    model_filename = (
        f"{model_name}_h{config['hidden_dim']}_d{config['dropout']}"
        f"_l{config['num_layers']}"
    )

    if "heads" in config:
        model_filename += f"_heads{config['heads']}"
    if "use_skip" in config:
        model_filename += f"_skip{config['use_skip']}"

    model_filename += f"_acc{config['best_val_acc']:.4f}.pt"

    model_path = os.path.join(MODEL_DIRS[model_name], model_filename)

    # 保存模型和训练历史
    save_dict = {
        "model_state_dict": model.state_dict(),
        "config": config,
        "best_val_acc": config["best_val_acc"],
        "test_acc": config["test_acc"],
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

    if training_history:
        save_dict["training_history"] = training_history

    torch.save(save_dict, model_path)

    # 准备Excel行数据
    result_row = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "hidden_dim": config["hidden_dim"],
        "dropout": config["dropout"],
        "num_layers": config["num_layers"],
        "best_val_acc": config["best_val_acc"],
        "test_acc": config["test_acc"],
        "test_f1_macro": config["test_f1_macro"],
        "test_f1_weighted": config["test_f1_weighted"],
        "test_precision_macro": config.get("test_precision_macro", 0),
        "test_recall_macro": config.get("test_recall_macro", 0),
        "num_params": config.get("num_params", 0),
        "model_file": model_filename,
    }

    if "heads" in config:
        result_row["heads"] = config["heads"]
    if "use_skip" in config:
        result_row["use_skip"] = config["use_skip"]

    # 保存到Excel
    excel_path = EXCEL_PATHS[model_name]

    if os.path.exists(excel_path):
        try:
            existing_df = pd.read_excel(excel_path)
            results_df = pd.concat(
                [existing_df, pd.DataFrame([result_row])], ignore_index=True
            )
        except:
            results_df = pd.DataFrame([result_row])
    else:
        results_df = pd.DataFrame([result_row])

    results_df = results_df.sort_values("best_val_acc", ascending=False)
    results_df.to_excel(excel_path, index=False)

    print(f"  ✓ 模型已保存: {model_path}")
    return model_path


def plot_training_curves(
    train_losses,
    val_losses,
    train_accs,
    val_accs,
    model_name,
    params,
    save_dir=CURVES_DIR,
):
    """绘制详细的训练曲线"""

    # 创建子图
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # 1. 损失曲线
    axes[0, 0].plot(train_losses, label="Train Loss", alpha=0.7, linewidth=2)
    axes[0, 0].plot(val_losses, label="Val Loss", alpha=0.7, linewidth=2)
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("Loss")
    axes[0, 0].set_title(f"{model_name} - Loss Curves")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # 2. 准确率曲线
    axes[0, 1].plot(train_accs, label="Train Acc", alpha=0.7, linewidth=2)
    axes[0, 1].plot(val_accs, label="Val Acc", alpha=0.7, linewidth=2)
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("Accuracy")
    axes[0, 1].set_title(f"{model_name} - Accuracy Curves")
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # 3. 损失差异曲线（过拟合分析）
    loss_diff = np.array(train_losses) - np.array(val_losses)
    axes[1, 0].plot(loss_diff, color="red", alpha=0.7, linewidth=1.5)
    axes[1, 0].axhline(y=0, color="black", linestyle="--", alpha=0.5)
    axes[1, 0].fill_between(range(len(loss_diff)), 0, loss_diff, alpha=0.3, color="red")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].set_ylabel("Train Loss - Val Loss")
    axes[1, 0].set_title("Overfitting Analysis (Loss Difference)")
    axes[1, 0].grid(True, alpha=0.3)

    # 4. 最佳验证准确率标记
    best_val_acc = max(val_accs)
    best_epoch = val_accs.index(best_val_acc)
    axes[1, 1].plot(val_accs, label="Val Acc", alpha=0.7, linewidth=2)
    axes[1, 1].scatter(
        best_epoch,
        best_val_acc,
        color="red",
        s=100,
        zorder=5,
        label=f"Best: {best_val_acc:.4f} (epoch {best_epoch})",
    )
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].set_ylabel("Validation Accuracy")
    axes[1, 1].set_title(f"Best Validation Accuracy: {best_val_acc:.4f}")
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.suptitle(
        f"{model_name} Training Analysis\nParameters: {params}",
        fontsize=14,
        fontweight="bold",
    )
    plt.tight_layout()

    # 保存图片
    param_str = f"h{params.get('hidden_dim', 0)}_d{params.get('dropout', 0)}_l{params.get('num_layers', 0)}"
    if "heads" in params:
        param_str += f"_heads{params['heads']}"
    filename = f"{model_name}_{param_str}_acc{best_val_acc:.4f}.png"
    save_path = os.path.join(save_dir, filename)
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close()

    return save_path


def plot_confusion_matrix(
    true_labels, pred_labels, class_names, model_name, params, save_dir=CONFUSION_DIR
):
    """绘制混淆矩阵"""

    cm = confusion_matrix(true_labels, pred_labels)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # 1. 原始混淆矩阵
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0])
    axes[0].set_xlabel("Predicted Label")
    axes[0].set_ylabel("True Label")
    axes[0].set_title(f"{model_name} - Confusion Matrix (Counts)")

    # 2. 归一化混淆矩阵
    cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="RdBu_r", ax=axes[1])
    axes[1].set_xlabel("Predicted Label")
    axes[1].set_ylabel("True Label")
    axes[1].set_title(f"{model_name} - Confusion Matrix (Normalized)")

    plt.suptitle(
        f"{model_name} Confusion Matrices\nParameters: {params}",
        fontsize=14,
        fontweight="bold",
    )
    plt.tight_layout()

    # 保存图片
    param_str = f"h{params.get('hidden_dim', 0)}_d{params.get('dropout', 0)}_l{params.get('num_layers', 0)}"
    if "heads" in params:
        param_str += f"_heads{params['heads']}"
    filename = f"{model_name}_{param_str}_confusion.png"
    save_path = os.path.join(save_dir, filename)
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close()

    return save_path, cm, cm_norm


def save_detailed_results(
    model_name, config, training_history, test_true, test_pred, class_names
):
    """保存详细结果到JSON文件"""

    # 计算各类别指标
    precision, recall, f1, support = precision_recall_fscore_support(
        test_true, test_pred, average=None
    )

    detailed_results = {
        "model_name": model_name,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "config": config,
        "best_val_acc": config["best_val_acc"],
        "test_metrics": {
            "accuracy": config["test_acc"],
            "f1_macro": config["test_f1_macro"],
            "f1_weighted": config["test_f1_weighted"],
            "precision_macro": config.get("test_precision_macro", 0),
            "recall_macro": config.get("test_recall_macro", 0),
        },
        "per_class_metrics": {
            f"class_{i}": {
                "class_name": class_names[i] if i < len(class_names) else f"Class_{i}",
                "precision": float(precision[i]),
                "recall": float(recall[i]),
                "f1": float(f1[i]),
                "support": int(support[i]),
            }
            for i in range(len(class_names))
        },
        "training_history_summary": {
            "final_train_loss": float(training_history["train_losses"][-1]),
            "final_val_loss": float(training_history["val_losses"][-1]),
            "best_val_acc": config["best_val_acc"],
            "best_val_loss": min(training_history["val_losses"]),
            "epochs_trained": len(training_history["train_losses"]),
        },
    }

    # 保存JSON
    param_str = f"h{config['hidden_dim']}_d{config['dropout']}_l{config['num_layers']}"
    if "heads" in config:
        param_str += f"_heads{config['heads']}"
    filename = f"{model_name}_{param_str}_acc{config['best_val_acc']:.4f}.json"
    save_path = os.path.join(MODEL_DIRS[model_name], filename)

    with open(save_path, "w") as f:
        json.dump(detailed_results, f, indent=2)

    return save_path


# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

# 设置设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

if torch.cuda.is_available():
    print(f"CUDA 版本: {torch.version.cuda}")
    print(f"GPU 名称: {torch.cuda.get_device_name()}")
    print(
        f"总显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

In [ ]:
# ============================================
# 1. 加载数据
# ============================================
print("\n" + "=" * 60)
print("1. 加载数据")
print("=" * 60)

df = pd.read_csv("./data/xenium_for_gnn.csv")
print(f"数据形状: {df.shape}")

# 删除缺失值
df = df.dropna()
print(f"删除缺失值后数据形状: {df.shape}")

In [ ]:
# ============================================
# 2. 提取特征和标签
# ============================================
print("\n" + "=" * 60)
print("2. 提取特征和标签")
print("=" * 60)

# 提取空间坐标
coords = df[["x_centroid", "y_centroid"]].values
print(f"坐标矩阵形状: {coords.shape}")

# 提取PCA特征
pca_cols = [col for col in df.columns if col.startswith("PC_")]
features = df[pca_cols].values
print(f"PCA特征形状: {features.shape}")

# 标准化特征
scaler = StandardScaler()
features = scaler.fit_transform(features)

# 提取标签
labels_raw = df["predicted_cell_type"].values
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(labels_raw)
num_classes = len(label_encoder.classes_)
print(f"类别数: {num_classes}")

# 获取类别名称
class_names = label_encoder.classes_.tolist() 
print(f"类别名称: {class_names}")

# 统计各类别样本数
class_counts = pd.Series(labels).value_counts().sort_index()
print(f"各类别样本数:")
for i, count in class_counts.items():
    print(f"  类别 {i}: {count}")

In [ ]:
# ============================================
# 3. 构建空间邻接图（使用较小的k值避免内存问题）
# ============================================
print("\n" + "=" * 60)
print("3. 构建空间邻接图")
print("=" * 60)


def build_spatial_graph(coords, k=10):
    """构建空间邻接图"""
    adj_matrix = kneighbors_graph(
        coords, n_neighbors=k, mode="connectivity", include_self=False
    )
    edge_index, _ = from_scipy_sparse_matrix(adj_matrix)
    return edge_index


# 尝试不同的k值
k_values = [5, 10, 15]
for k in k_values:
    edge_index = build_spatial_graph(coords, k=k)
    print(
        f"k={k}: 边数量={edge_index.shape[1]}, 平均度={2 * edge_index.shape[1] / coords.shape[0]:.2f}"
    )

# 使用较小的k值减少内存占用
DEFAULT_K = 5
edge_index = build_spatial_graph(coords, k=DEFAULT_K)
print(f"\n使用 k={DEFAULT_K} 构建图")
print(f"边数量: {edge_index.shape[1]}")

In [ ]:
# ============================================
# 4. 创建PyTorch Geometric数据对象
# ============================================
print("\n" + "=" * 60)
print("4. 创建PyTorch Geometric数据对象")
print("=" * 60)

# 转换为Tensor
x = torch.FloatTensor(features)
y = torch.LongTensor(labels)
edge_index = edge_index.to(torch.long)

# 创建数据对象
data = Data(x=x, edge_index=edge_index, y=y)

print(f"节点数: {data.num_nodes}")
print(f"特征维度: {data.num_features}")
print(f"边数: {data.num_edges}")
print(f"类别数: {num_classes}")

In [ ]:
# ============================================
# 5. 划分数据集
# ============================================
print("\n" + "=" * 60)
print("5. 划分数据集")
print("=" * 60)

indices = np.arange(data.num_nodes)
train_val_idx, test_idx = train_test_split(
    indices, test_size=0.2, stratify=labels, random_state=42
)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=0.25, stratify=labels[train_val_idx], random_state=42
)

train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
val_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
test_mask = torch.zeros(data.num_nodes, dtype=torch.bool)

train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print(f"训练集大小: {train_mask.sum().item()}")
print(f"验证集大小: {val_mask.sum().item()}")
print(f"测试集大小: {test_mask.sum().item()}")

In [ ]:
# ============================================
# 6. 定义模型
# ============================================
print("\n" + "=" * 60)
print("6. 定义图神经网络模型")
print("=" * 60)


class MLPWithDropout(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.3, num_layers=2):
        super(MLPWithDropout, self).__init__()
        self.layers = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.layers.append(nn.Linear(in_dim, hidden_dim))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        for i in range(num_layers - 2):
            self.layers.append(nn.Linear(hidden_dim, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.layers.append(nn.Linear(hidden_dim, out_dim))
        self.dropout = dropout

    def forward(self, data):
        x = data.x
        for i, layer in enumerate(self.layers[:-1]):
            x = layer(x)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.layers[-1](x)
        return F.log_softmax(x, dim=1)


class ImprovedGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.3, num_layers=2):
        super(ImprovedGCN, self).__init__()
        self.num_layers = num_layers
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.convs.append(GCNConv(in_dim, hidden_dim))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        for i in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.convs.append(GCNConv(hidden_dim, out_dim))
        self.dropout = dropout
        self.skip_proj = nn.Linear(in_dim, hidden_dim) if num_layers >= 2 else None

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        identity = x
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            if i == 0 and self.skip_proj is not None:
                x = x + self.skip_proj(identity)
        x = self.convs[-1](x, edge_index)
        return F.log_softmax(x, dim=1)


class GraphSAGEWithNorm(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.3, num_layers=2):
        super(GraphSAGEWithNorm, self).__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.convs.append(SAGEConv(in_dim, hidden_dim))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        for i in range(num_layers - 2):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.convs.append(SAGEConv(hidden_dim, out_dim))
        self.dropout = dropout

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.convs[-1](x, edge_index)
        return F.log_softmax(x, dim=1)


class GAT(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden_dim,
        out_dim,
        heads=8,
        dropout=0.3,
        num_layers=2,
        use_skip=True,
    ):
        super(GAT, self).__init__()
        self.num_layers = num_layers
        self.use_skip = use_skip
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.convs.append(
            GATConv(in_dim, hidden_dim, heads=heads, dropout=dropout, concat=True)
        )
        self.bns.append(nn.BatchNorm1d(hidden_dim * heads))
        for i in range(num_layers - 2):
            self.convs.append(
                GATConv(
                    hidden_dim * heads,
                    hidden_dim,
                    heads=heads,
                    dropout=dropout,
                    concat=True,
                )
            )
            self.bns.append(nn.BatchNorm1d(hidden_dim * heads))
        self.convs.append(
            GATConv(hidden_dim * heads, out_dim, heads=1, dropout=dropout, concat=False)
        )
        self.dropout = dropout
        if use_skip and num_layers >= 2:
            self.skip_proj = nn.Linear(in_dim, hidden_dim * heads)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        if self.use_skip and self.num_layers >= 2:
            identity = self.skip_proj(x)
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x, edge_index)
            x = self.bns[i](x)
            x = F.elu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            if self.use_skip and i == 0 and self.num_layers >= 2:
                x = x + identity
        x = self.convs[-1](x, edge_index)
        return F.log_softmax(x, dim=1)


class GATv2Model(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, heads=8, dropout=0.3, num_layers=2):
        super(GATv2Model, self).__init__()
        self.num_layers = num_layers
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.convs.append(
            GATv2Conv(in_dim, hidden_dim, heads=heads, dropout=dropout, concat=True)
        )
        self.bns.append(nn.BatchNorm1d(hidden_dim * heads))
        for i in range(num_layers - 2):
            self.convs.append(
                GATv2Conv(
                    hidden_dim * heads,
                    hidden_dim,
                    heads=heads,
                    dropout=dropout,
                    concat=True,
                )
            )
            self.bns.append(nn.BatchNorm1d(hidden_dim * heads))
        self.convs.append(
            GATv2Conv(
                hidden_dim * heads, out_dim, heads=1, dropout=dropout, concat=False
            )
        )
        self.dropout = dropout

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x, edge_index)
            x = self.bns[i](x)
            x = F.elu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.convs[-1](x, edge_index)
        return F.log_softmax(x, dim=1)


print("所有模型定义完成")

In [ ]:
# ============================================
# 7. 训练和评估函数
# ============================================
print("\n" + "=" * 60)
print("7. 定义训练和评估函数")
print("=" * 60)


def train_model(model, data, optimizer, epochs=500, patience=50, verbose=True):
    """训练模型并返回详细历史"""
    data = data.to(device)
    model = model.to(device)

    scaler = torch.cuda.amp.GradScaler()

    best_val_acc = 0
    best_val_loss = float("inf")
    best_model_state = None
    patience_counter = 0

    train_losses, val_losses = [], []
    train_accs, val_accs = [], []

    for epoch in range(epochs):
        # 训练
        model.train()
        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            out = model(data)
            loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        # 计算训练指标
        with torch.no_grad():
            pred = out.argmax(dim=1)
            train_acc = (
                (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            )

        # 验证
        model.eval()
        with torch.no_grad():
            out = model(data)
            val_loss = F.nll_loss(out[data.val_mask], data.y[data.val_mask])
            pred = out.argmax(dim=1)
            val_acc = (
                (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            )

        train_losses.append(loss.item())
        val_losses.append(val_loss.item())
        train_accs.append(train_acc)
        val_accs.append(val_acc)

        # 早停检查
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            if verbose:
                print(f"  早停于 epoch {epoch+1}")
            break

        if verbose and (epoch + 1) % 50 == 0:
            print(
                f"  Epoch {epoch+1:4d}/{epochs} | Train Loss: {loss.item():.4f} | "
                f"Train Acc: {train_acc:.4f} | Val Loss: {val_loss.item():.4f} | Val Acc: {val_acc:.4f}"
            )

        if (epoch + 1) % 100 == 0:
            clear_memory()

    # 加载最佳模型
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return model, train_losses, val_losses, train_accs, val_accs, best_val_acc


def evaluate_model(model, data):
    """详细评估模型"""
    model.eval()
    data = data.to(device)

    with torch.no_grad():
        out = model(data)
        pred = out.argmax(dim=1)

        test_pred = pred[data.test_mask].cpu().numpy()
        test_true = data.y[data.test_mask].cpu().numpy()

        test_acc = accuracy_score(test_true, test_pred)
        test_f1_macro = f1_score(test_true, test_pred, average="macro")
        test_f1_weighted = f1_score(test_true, test_pred, average="weighted")

        precision, recall, f1, support = precision_recall_fscore_support(
            test_true, test_pred, average="macro"
        )

        return {
            "test_acc": test_acc,
            "test_f1_macro": test_f1_macro,
            "test_f1_weighted": test_f1_weighted,
            "test_precision_macro": precision,
            "test_recall_macro": recall,
            "test_pred": test_pred,
            "test_true": test_true,
        }


def clear_memory():
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    gc.collect()

In [ ]:
# ============================================
# 8. 模型参数实验配置
# ============================================
print("\n" + "=" * 60)
print("8. 模型参数实验配置")
print("=" * 60)

in_dim = data.num_features
out_dim = num_classes

param_grids = {
    "MLP": {
        "hidden_dim": [128, 256],
        "dropout": [0.2, 0.25, 0.3],
        "num_layers": [2, 3, 4],
    },
    "GCN": {
        "hidden_dim": [128, 256],
        "dropout": [0.2, 0.25, 0.3],
        "num_layers": [2, 3, 4],
    },
    "GraphSAGE": {
        "hidden_dim": [128, 256],
        "dropout": [0.2, 0.25, 0.3],
        "num_layers": [2, 3, 4],
    },
    "GAT": {
        "hidden_dim": [64, 128],
        "heads": [4, 8],
        "dropout": [0.2, 0.25, 0.3],
        "num_layers": [2, 3],
        "use_skip": [True],
    },
    "GATv2": {
        "hidden_dim": [64, 128],
        "heads": [4, 8],
        "dropout": [0.2, 0.25, 0.3],
        "num_layers": [2, 3],
    },
}

In [ ]:
# ============================================
# 9.1 MLP参数实验
# ============================================
print("\n" + "=" * 50)
print("9.1 MLP 参数实验")
print("=" * 50)

mlp_results = []
param_combinations = list(
    product(
        param_grids["MLP"]["hidden_dim"],
        param_grids["MLP"]["dropout"],
        param_grids["MLP"]["num_layers"],
    )
)

for idx, (hidden_dim, dropout, num_layers) in enumerate(param_combinations):
    print(
        f"\n[{idx+1}/{len(param_combinations)}] 训练 MLP (hidden_dim={hidden_dim}, dropout={dropout}, num_layers={num_layers})"
    )

    model = MLPWithDropout(in_dim, hidden_dim, out_dim, dropout, num_layers)
    num_params = sum(p.numel() for p in model.parameters())
    print(f"  参数量: {num_params:,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

    try:
        trained_model, train_losses, val_losses, train_accs, val_accs, best_val_acc = (
            train_model(model, data, optimizer, epochs=500, patience=50, verbose=False)
        )

        eval_results = evaluate_model(trained_model, data)

        config = {
            "hidden_dim": hidden_dim,
            "dropout": dropout,
            "num_layers": num_layers,
            "best_val_acc": best_val_acc,
            "num_params": num_params,
            **eval_results,
        }

        training_history = {
            "train_losses": train_losses,
            "val_losses": val_losses,
            "train_accs": train_accs,
            "val_accs": val_accs,
        }

        # 保存模型和结果
        save_model_and_results(trained_model, config, "MLP", training_history)

        # 绘制训练曲线
        plot_training_curves(
            train_losses, val_losses, train_accs, val_accs, "MLP", config, CURVES_DIR
        )

        # 绘制混淆矩阵
        plot_confusion_matrix(
            eval_results["test_true"],
            eval_results["test_pred"],
            class_names,
            "MLP",
            config,
            CONFUSION_DIR,
        )

        # 保存详细结果
        save_detailed_results(
            "MLP",
            config,
            training_history,
            eval_results["test_true"],
            eval_results["test_pred"],
            class_names,
        )

        mlp_results.append(config)

        print(
            f"  ✓ 验证准确率: {best_val_acc:.4f} | 测试准确率: {eval_results['test_acc']:.4f} | F1: {eval_results['test_f1_macro']:.4f}"
        )

    except Exception as e:
        print(f"  ✗ 训练失败: {e}")

    finally:
        del model, trained_model, optimizer
        clear_memory()

if mlp_results:
    best_mlp = max(mlp_results, key=lambda x: x["best_val_acc"])
    print(
        f"\nMLP 最佳模型: hidden_dim={best_mlp['hidden_dim']}, dropout={best_mlp['dropout']}, "
        f"num_layers={best_mlp['num_layers']}, Test Acc={best_mlp['test_acc']:.4f}"
    )

In [ ]:
# ============================================
# 9.2 GCN参数实验
# ============================================
print("\n" + "=" * 50)
print("9.2 GCN 参数实验")
print("=" * 50)

gcn_results = []
param_combinations = list(
    product(
        param_grids["GCN"]["hidden_dim"],
        param_grids["GCN"]["dropout"],
        param_grids["GCN"]["num_layers"],
    )
)

for idx, (hidden_dim, dropout, num_layers) in enumerate(param_combinations):
    print(
        f"\n[{idx+1}/{len(param_combinations)}] 训练 GCN (hidden_dim={hidden_dim}, dropout={dropout}, num_layers={num_layers})"
    )

    model = ImprovedGCN(in_dim, hidden_dim, out_dim, dropout, num_layers)
    num_params = sum(p.numel() for p in model.parameters())
    print(f"  参数量: {num_params:,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

    try:
        trained_model, train_losses, val_losses, train_accs, val_accs, best_val_acc = (
            train_model(model, data, optimizer, epochs=500, patience=50, verbose=False)
        )

        eval_results = evaluate_model(trained_model, data)

        config = {
            "hidden_dim": hidden_dim,
            "dropout": dropout,
            "num_layers": num_layers,
            "best_val_acc": best_val_acc,
            "num_params": num_params,
            **eval_results,
        }

        training_history = {
            "train_losses": train_losses,
            "val_losses": val_losses,
            "train_accs": train_accs,
            "val_accs": val_accs,
        }

        save_model_and_results(trained_model, config, "GCN", training_history)
        plot_training_curves(
            train_losses,
            val_losses,
            train_accs,
            val_accs,
            "GCN",
            config,
            CURVES_DIR,
        )
        plot_confusion_matrix(
            eval_results["test_true"],
            eval_results["test_pred"],
            class_names,
            "GCN",
            config,
            CONFUSION_DIR,
        )
        save_detailed_results(
            "GCN",
            config,
            training_history,
            eval_results["test_true"],
            eval_results["test_pred"],
            class_names,
        )

        gcn_results.append(config)

        print(
            f"  ✓ 验证准确率: {best_val_acc:.4f} | 测试准确率: {eval_results['test_acc']:.4f} | F1: {eval_results['test_f1_macro']:.4f}"
        )

    except Exception as e:
        print(f"  ✗ 训练失败: {e}")

    finally:
        del model, trained_model, optimizer
        clear_memory()

if gcn_results:
    best_gcn = max(gcn_results, key=lambda x: x["best_val_acc"])
    print(
        f"\nGCN 最佳模型: hidden_dim={best_gcn['hidden_dim']}, dropout={best_gcn['dropout']}, "
        f"num_layers={best_gcn['num_layers']}, Test Acc={best_gcn['test_acc']:.4f}"
    )

In [ ]:
# ============================================
# 9.3 GraphSAGE参数实验
# ============================================
print("\n" + "=" * 50)
print("9.3 GraphSAGE 参数实验")
print("=" * 50)

sage_results = []
param_combinations = list(
    product(
        param_grids["GraphSAGE"]["hidden_dim"],
        param_grids["GraphSAGE"]["dropout"],
        param_grids["GraphSAGE"]["num_layers"],
    )
)

for idx, (hidden_dim, dropout, num_layers) in enumerate(param_combinations):
    print(
        f"\n[{idx+1}/{len(param_combinations)}] 训练 GraphSAGE (hidden_dim={hidden_dim}, dropout={dropout}, num_layers={num_layers})"
    )

    model = GraphSAGEWithNorm(in_dim, hidden_dim, out_dim, dropout, num_layers)
    num_params = sum(p.numel() for p in model.parameters())
    print(f"  参数量: {num_params:,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

    try:
        trained_model, train_losses, val_losses, train_accs, val_accs, best_val_acc = (
            train_model(model, data, optimizer, epochs=500, patience=50, verbose=False)
        )

        eval_results = evaluate_model(trained_model, data)

        config = {
            "hidden_dim": hidden_dim,
            "dropout": dropout,
            "num_layers": num_layers,
            "best_val_acc": best_val_acc,
            "num_params": num_params,
            **eval_results,
        }

        training_history = {
            "train_losses": train_losses,
            "val_losses": val_losses,
            "train_accs": train_accs,
            "val_accs": val_accs,
        }

        save_model_and_results(trained_model, config, "GraphSAGE", training_history)
        plot_training_curves(
            train_losses,
            val_losses,
            train_accs,
            val_accs,
            "GraphSAGE",
            config,
            CURVES_DIR,
        )
        plot_confusion_matrix(
            eval_results["test_true"],
            eval_results["test_pred"],
            class_names,
            "GraphSAGE",
            config,
            CONFUSION_DIR,
        )
        save_detailed_results(
            "GraphSAGE",
            config,
            training_history,
            eval_results["test_true"],
            eval_results["test_pred"],
            class_names,
        )

        sage_results.append(config)

        print(
            f"  ✓ 验证准确率: {best_val_acc:.4f} | 测试准确率: {eval_results['test_acc']:.4f} | F1: {eval_results['test_f1_macro']:.4f}"
        )

    except Exception as e:
        print(f"  ✗ 训练失败: {e}")

    finally:
        del model, trained_model, optimizer
        clear_memory()

if sage_results:
    best_sage = max(sage_results, key=lambda x: x["best_val_acc"])
    print(
        f"\nGraphSAGE 最佳模型: hidden_dim={best_sage['hidden_dim']}, dropout={best_sage['dropout']}, "
        f"num_layers={best_sage['num_layers']}, Test Acc={best_sage['test_acc']:.4f}"
    )

In [ ]:
# ============================================
# 9.4 GAT参数实验
# ============================================
print("\n" + "=" * 50)
print("9.4 GAT 参数实验")
print("=" * 50)

gat_results = []
param_combinations = list(
    product(
        param_grids["GAT"]["hidden_dim"],
        param_grids["GAT"]["heads"],
        param_grids["GAT"]["dropout"],
        param_grids["GAT"]["num_layers"],
        param_grids["GAT"]["use_skip"],
    )
)

for idx, (hidden_dim, heads, dropout, num_layers, use_skip) in enumerate(
    param_combinations
):
    print(
        f"\n[{idx+1}/{len(param_combinations)}] 训练 GAT (hidden_dim={hidden_dim}, heads={heads}, "
        f"dropout={dropout}, num_layers={num_layers}, use_skip={use_skip})"
    )

    model = GAT(in_dim, hidden_dim, out_dim, heads, dropout, num_layers, use_skip)
    num_params = sum(p.numel() for p in model.parameters())
    print(f"  参数量: {num_params:,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

    try:
        trained_model, train_losses, val_losses, train_accs, val_accs, best_val_acc = (
            train_model(model, data, optimizer, epochs=500, patience=50, verbose=False)
        )

        eval_results = evaluate_model(trained_model, data)

        config = {
            "hidden_dim": hidden_dim,
            "heads": heads,
            "dropout": dropout,
            "num_layers": num_layers,
            "use_skip": use_skip,
            "best_val_acc": best_val_acc,
            "num_params": num_params,
            **eval_results,
        }

        training_history = {
            "train_losses": train_losses,
            "val_losses": val_losses,
            "train_accs": train_accs,
            "val_accs": val_accs,
        }

        save_model_and_results(trained_model, config, "GAT", training_history)
        plot_training_curves(
            train_losses, val_losses, train_accs, val_accs, "GAT", config, CURVES_DIR
        )
        plot_confusion_matrix(
            eval_results["test_true"],
            eval_results["test_pred"],
            class_names,
            "GAT",
            config,
            CONFUSION_DIR,
        )
        save_detailed_results(
            "GAT",
            config,
            training_history,
            eval_results["test_true"],
            eval_results["test_pred"],
            class_names,
        )

        gat_results.append(config)

        print(
            f"  ✓ 验证准确率: {best_val_acc:.4f} | 测试准确率: {eval_results['test_acc']:.4f} | F1: {eval_results['test_f1_macro']:.4f}"
        )

    except Exception as e:
        print(f"  ✗ 训练失败: {e}")

    finally:
        del model, trained_model, optimizer
        clear_memory()

if gat_results:
    best_gat = max(gat_results, key=lambda x: x["best_val_acc"])
    print(
        f"\nGAT 最佳模型: hidden_dim={best_gat['hidden_dim']}, heads={best_gat['heads']}, "
        f"dropout={best_gat['dropout']}, num_layers={best_gat['num_layers']}, "
        f"Test Acc={best_gat['test_acc']:.4f}"
    )

In [ ]:
# ============================================
# 9.5 GATv2参数实验
# ============================================
print("\n" + "=" * 50)
print("9.5 GATv2 参数实验")
print("=" * 50)

gatv2_results = []
param_combinations = list(
    product(
        param_grids["GATv2"]["hidden_dim"],
        param_grids["GATv2"]["heads"],
        param_grids["GATv2"]["dropout"],
        param_grids["GATv2"]["num_layers"],
    )
)

for idx, (hidden_dim, heads, dropout, num_layers) in enumerate(param_combinations):
    print(
        f"\n[{idx+1}/{len(param_combinations)}] 训练 GATv2 (hidden_dim={hidden_dim}, heads={heads}, "
        f"dropout={dropout}, num_layers={num_layers})"
    )

    model = GATv2Model(in_dim, hidden_dim, out_dim, heads, dropout, num_layers)
    num_params = sum(p.numel() for p in model.parameters())
    print(f"  参数量: {num_params:,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

    try:
        trained_model, train_losses, val_losses, train_accs, val_accs, best_val_acc = (
            train_model(model, data, optimizer, epochs=500, patience=50, verbose=False)
        )

        eval_results = evaluate_model(trained_model, data)

        config = {
            "hidden_dim": hidden_dim,
            "heads": heads,
            "dropout": dropout,
            "num_layers": num_layers,
            "best_val_acc": best_val_acc,
            "num_params": num_params,
            **eval_results,
        }

        training_history = {
            "train_losses": train_losses,
            "val_losses": val_losses,
            "train_accs": train_accs,
            "val_accs": val_accs,
        }

        save_model_and_results(trained_model, config, "GATv2", training_history)
        plot_training_curves(
            train_losses, val_losses, train_accs, val_accs, "GATv2", config, CURVES_DIR
        )
        plot_confusion_matrix(
            eval_results["test_true"],
            eval_results["test_pred"],
            class_names,
            "GATv2",
            config,
            CONFUSION_DIR,
        )
        save_detailed_results(
            "GATv2",
            config,
            training_history,
            eval_results["test_true"],
            eval_results["test_pred"],
            class_names,
        )

        gatv2_results.append(config)

        print(
            f"  ✓ 验证准确率: {best_val_acc:.4f} | 测试准确率: {eval_results['test_acc']:.4f} | F1: {eval_results['test_f1_macro']:.4f}"
        )

    except Exception as e:
        print(f"  ✗ 训练失败: {e}")

    finally:
        del model, trained_model, optimizer
        clear_memory()

if gatv2_results:
    best_gatv2 = max(gatv2_results, key=lambda x: x["best_val_acc"])
    print(
        f"\nGATv2 最佳模型: hidden_dim={best_gatv2['hidden_dim']}, heads={best_gatv2['heads']}, "
        f"dropout={best_gatv2['dropout']}, num_layers={best_gatv2['num_layers']}, "
        f"Test Acc={best_gatv2['test_acc']:.4f}"
    )

In [ ]:
# ============================================
# 10. 结果汇总和可视化
# ============================================
print("\n" + "=" * 60)
print("10. 结果汇总和可视化")
print("=" * 60)

# 收集所有最佳模型
all_best_models = {}
for model_name, results in [
    ("MLP", mlp_results),
    ("GCN", gcn_results),
    ("GraphSAGE", sage_results),
    ("GAT", gat_results),
    ("GATv2", gatv2_results),
]:
    if results:
        all_best_models[model_name] = max(results, key=lambda x: x["best_val_acc"])

# 创建汇总表
if all_best_models:
    summary_data = []
    for model_name, best_model in all_best_models.items():
        summary_data.append(
            {
                "Model": model_name,
                "Best Val Acc": best_model["best_val_acc"],
                "Test Acc": best_model["test_acc"],
                "Test F1 (Macro)": best_model["test_f1_macro"],
                "Test F1 (Weighted)": best_model["test_f1_weighted"],
                "Precision (Macro)": best_model.get("test_precision_macro", 0),
                "Recall (Macro)": best_model.get("test_recall_macro", 0),
                "Hidden Dim": best_model["hidden_dim"],
                "Dropout": best_model["dropout"],
                "Layers": best_model["num_layers"],
                "Heads": best_model.get("heads", "N/A"),
                "Params": best_model.get("num_params", 0),
            }
        )

    summary_df = pd.DataFrame(summary_data)
    summary_df = summary_df.sort_values("Test Acc", ascending=False)

    print("\n模型性能对比表:")
    print(summary_df.to_string(index=False))

    # 保存汇总表
    summary_df.to_csv(os.path.join(RESULTS_DIR, "model_summary.csv"), index=False)
    summary_df.to_excel(os.path.join(RESULTS_DIR, "model_summary.xlsx"), index=False)
    print(f"\n✓ 汇总表已保存到 {RESULTS_DIR}")

    # 绘制详细对比图
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    # 1. 测试准确率对比
    ax1 = axes[0, 0]
    bars = ax1.bar(summary_df["Model"], summary_df["Test Acc"], color="steelblue")
    ax1.set_ylabel("Accuracy")
    ax1.set_title("Model Test Accuracy Comparison")
    ax1.set_ylim(0, 1)
    for bar, acc in zip(bars, summary_df["Test Acc"]):
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{acc:.4f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )
    ax1.grid(True, alpha=0.3)

    # 2. F1分数对比
    ax2 = axes[0, 1]
    x = np.arange(len(summary_df))
    width = 0.35
    ax2.bar(
        x - width / 2,
        summary_df["Test F1 (Macro)"],
        width,
        label="Macro F1",
        color="forestgreen",
    )
    ax2.bar(
        x + width / 2,
        summary_df["Test F1 (Weighted)"],
        width,
        label="Weighted F1",
        color="lightcoral",
    )
    ax2.set_xticks(x)
    ax2.set_xticklabels(summary_df["Model"], rotation=45, ha="right")
    ax2.set_ylabel("F1 Score")
    ax2.set_title("Model F1 Score Comparison")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    # 3. 参数量 vs 准确率
    ax3 = axes[1, 0]
    ax3.scatter(summary_df["Params"] / 1000, summary_df["Test Acc"], s=100, alpha=0.6)
    for _, row in summary_df.iterrows():
        ax3.annotate(
            row["Model"],
            (row["Params"] / 1000, row["Test Acc"]),
            fontsize=8,
            ha="center",
        )
    ax3.set_xlabel("Number of Parameters (K)")
    ax3.set_ylabel("Test Accuracy")
    ax3.set_title("Model Efficiency: Parameters vs Accuracy")
    ax3.grid(True, alpha=0.3)

    # 4. 雷达图 - 多指标对比
    ax4 = axes[1, 1]
    metrics = ["Test Acc", "Test F1 (Macro)", "Precision (Macro)", "Recall (Macro)"]
    num_metrics = len(metrics)
    angles = np.linspace(0, 2 * np.pi, num_metrics, endpoint=False).tolist()
    angles += angles[:1]

    for _, row in summary_df.iterrows():
        values = [row[m] for m in metrics]
        values += values[:1]
        ax4.plot(angles, values, "o-", linewidth=2, label=row["Model"], alpha=0.7)

    ax4.set_xticks(angles[:-1])
    ax4.set_xticklabels(metrics)
    ax4.set_ylim(0, 1)
    ax4.set_title("Multi-metric Radar Chart")
    ax4.legend(loc="upper right", bbox_to_anchor=(1.3, 1.0))
    ax4.grid(True)

    plt.suptitle(
        "Comprehensive Model Performance Comparison", fontsize=14, fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig(FIGURE_PATHS["test_comparison"], dpi=200, bbox_inches="tight")
    plt.show()

    # 找出最终最佳模型
    best_model_name = summary_df.iloc[0]["Model"]
    print(f"\n🏆 最终最佳模型: {best_model_name}")
    print(f"   测试准确率: {summary_df.iloc[0]['Test Acc']:.4f}")
    print(f"   Macro F1: {summary_df.iloc[0]['Test F1 (Macro)']:.4f}")

print("\n" + "=" * 60)
print("实验完成！")
print(f"所有结果已保存到: {BASE_DIR}")
print(f"  - 模型文件: {MODELS_DIR}")
print(f"  - 训练曲线: {CURVES_DIR}")
print(f"  - 混淆矩阵: {CONFUSION_DIR}")
print(f"  - 结果汇总: {RESULTS_DIR}")
print("=" * 60)